In [8]:
# ============================================================
# Google Drive 연결 및 프로젝트 루트 설정
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "코드잇_파트2_3팀_프로젝트/"
    "pill-object-detection"
)

os.chdir(PROJECT_ROOT)

print("현재 작업 디렉터리:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 작업 디렉터리: /content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/pill-object-detection


In [9]:
# ============================================================
# 저장된 Prediction / Mapping Load
# ============================================================

from pathlib import Path

import torch

PREDICTIONS_PATH = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "predictions"
    / "faster_rcnn_baseline_predictions_v1.pt"
)

MAPPING_PATH = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "predictions"
    / "faster_rcnn_label_to_category_id_v1.pt"
)

predictions = torch.load(
    PREDICTIONS_PATH,
    weights_only=False,
)

label_to_category_id = torch.load(
    MAPPING_PATH,
    weights_only=False,
)

print("prediction 수:", len(predictions))
print("mapping 수:", len(label_to_category_id))
print("mapping 예시:", list(label_to_category_id.items())[:10])

prediction 수: 842
mapping 수: 56
mapping 예시: [(1, 1900), (2, 2483), (3, 3351), (4, 3483), (5, 3544), (6, 3743), (7, 3832), (8, 4543), (9, 12081), (10, 12247)]


In [10]:
# ============================================================
# Inference Module Import
# ============================================================

from src.inference import (
    predictions_to_submission,
    save_submission,
)

In [11]:
# ============================================================
# Threshold Sweep
# ============================================================

from pathlib import Path

THRESHOLDS = [
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
]

SUBMISSION_DIR = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "submissions"
)

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for threshold in THRESHOLDS:

    submission_df = predictions_to_submission(
        predictions=predictions,
        score_threshold=threshold,
        label_to_category_id=label_to_category_id,
    )

    threshold_name = f"{int(threshold * 100):03d}"

    output_path = (
        SUBMISSION_DIR
        / f"faster_rcnn_baseline_thr{threshold_name}.csv"
    )

    save_submission(
        submission_df=submission_df,
        output_path=output_path,
    )

    print(
        f"threshold={threshold:.2f} | "
        f"rows={len(submission_df):,} | "
        f"images={submission_df['image_id'].nunique()} | "
        f"file={output_path.name}"
    )

threshold=0.05 | rows=19,028 | images=842 | file=faster_rcnn_baseline_thr005.csv
threshold=0.10 | rows=11,339 | images=842 | file=faster_rcnn_baseline_thr010.csv
threshold=0.20 | rows=6,143 | images=842 | file=faster_rcnn_baseline_thr020.csv
threshold=0.30 | rows=4,155 | images=842 | file=faster_rcnn_baseline_thr030.csv
threshold=0.40 | rows=3,059 | images=840 | file=faster_rcnn_baseline_thr040.csv
threshold=0.50 | rows=2,347 | images=837 | file=faster_rcnn_baseline_thr050.csv
